In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Generate Synthetic Sequential Data (Sine Wave with Noise)
np.random.seed(42)
time_steps = np.linspace(0, 50, 400)
data = np.sin(time_steps) + np.random.normal(scale=0.1, size=len(time_steps))

# Create input windows (X: past 10 steps) to predict next value (y: step 11)
window_size = 10
X, y = [], []
for i in range(len(data) - window_size):
    X.append(data[i : i + window_size])
    y.append(data[i + window_size])

X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1) # Shape: (batch, 10, 1)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1) # Shape: (batch, 1)

# 2. Define Time-Series RNN Architecture
class TimeSeriesRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, output_size=1):
        super(TimeSeriesRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        out, _ = self.rnn(x) 
        
        # Take the output state of the LAST time step in the window
        last_step_out = out[:, -1, :] 
        
        # Predict the single continuous continuous scalar
        prediction = self.fc(last_step_out)
        return prediction

model = TimeSeriesRNN(input_size=1, hidden_size=16, output_size=1)
criterion = nn.MSELoss()  # Mean Squared Error for regression tasks
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 3. Training Loop
for epoch in range(1, 151):
    optimizer.zero_grad()
    
    predictions = model(X)
    loss = criterion(predictions, y)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 30 == 0:
        print(f"Epoch {epoch:3d}/150 | MSE Loss: {loss.item():.5f}")

# 4. Make a Single Forecast Prediction
sample_window = X[0:1] # Past 10 steps: shape (1, 10, 1)
actual_next_value = y[0].item()

model.eval()
with torch.no_grad():
    predicted_next_value = model(sample_window).item()

print("\n--- Model Forecast Result ---")
print(f"Actual Next Value:    {actual_next_value:.4f}")
print(f"Predicted Next Value: {predicted_next_value:.4f}")